# 🖐️ GROOPY — Fingerspelling CNN Bake-off
### CRISP-DM: Modeling + Evaluation

Train **4 image classifiers under one fixed protocol** on the Kaggle **ASL Alphabet** (87k images, 29 classes: A–Z + `del`/`nothing`/`space`), then score them and pick a winner.

| Candidate | Type |
|---|---|
| `cnn_scratch` | CNN built from scratch (our baseline) |
| `mobilenetv2` | ImageNet transfer learning (mobile-friendly) |
| `efficientnetb0` | ImageNet transfer learning (accuracy/size sweet-spot) |
| `resnet50` | ImageNet transfer learning (high-capacity reference) |

The winner is chosen by a **weighted scorecard** (accuracy + latency + size + robustness + stability) — the most *shippable* model, not just the most accurate.

> ⚙️ **Before running:** set a GPU — **Runtime → Change runtime type → T4 GPU**.

## 1 · Setup
Clone the repo and install the one extra dependency (TensorFlow / scikit-learn / matplotlib already ship with Colab).

In [ ]:
# Clone if needed, otherwise pull the latest
!git clone https://github.com/SpliiiiT/GROOPY.git 2>/dev/null || (cd GROOPY && git pull -q)
%cd /content/GROOPY
import sys; sys.path.insert(0, '/content/GROOPY')
!pip install -q kaggle

## 2 · Get the data
The download needs a Kaggle API key:
1. On [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create Legacy API Key** → downloads `kaggle.json`.
2. Accept the dataset terms once on the [ASL Alphabet page](https://www.kaggle.com/datasets/grassknoted/asl-alphabet).
3. Run the cell and **upload `kaggle.json`** when prompted (~1 GB download).

In [ ]:
from google.colab import files; files.upload()   # pick kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!python data/download_asl_alphabet.py

## 3 · Confirm the GPU
Training on CPU would be far too slow — make sure a T4 is attached.

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPU:', gpus or 'NONE — set Runtime > Change runtime type > T4')

## 4 · Build a fast training subset
ASL Alphabet has **3000 images per class** — highly redundant. Decoding all 87k JPEGs every epoch on Colab's 2 CPUs is the bottleneck (~15 min/epoch). We **symlink ~600 images/class** (instant, no extra disk) so epochs run in ~1–3 min while models still reach high accuracy. The bake-off stays fair because every candidate uses the *same* subset and protocol.

_Short on time? Lower `N`. Want the full run? Point `data_dir` at the original folder instead._

In [ ]:
import os, glob
SRC, DST, N = '/content/GROOPY/data/asl_alphabet_train', '/content/asl_subset', 600
for cls in sorted(os.listdir(SRC)):
    os.makedirs(f'{DST}/{cls}', exist_ok=True)
    for f in sorted(glob.glob(f'{SRC}/{cls}/*.jpg'))[:N]:
        link = f'{DST}/{cls}/{os.path.basename(f)}'
        if not os.path.exists(link):
            os.symlink(f, link)
print(f'subset ready: {N} imgs/class at {DST}')

## 5 · Train the bake-off
One fixed protocol for every model: same split, augmentation, batch size, and epochs. The three pretrained backbones add a 2-phase fine-tune (frozen head → unfreeze top layers at a lower LR). `EarlyStopping` trims models that plateau.

In [ ]:
from recognition.src.train import train_one
from recognition.src.data import make_datasets
from recognition.src import models as zoo

train_ds, val_ds, test_ds, class_names = make_datasets(data_dir='/content/asl_subset',
                                                       batch_size=32)
records = [train_one(name, epochs=10, train_ds=train_ds, val_ds=val_ds)
           for name in zoo.REGISTRY]
records

## 6 · Evaluate + weighted scorecard → winner
Scores every trained model on the held-out **test** set, then ranks by the weighted scorecard (accuracy 40 %, latency 20 %, size 15 %, robustness 15 %, stability 10 %). Rank 1 is the winner.

In [ ]:
from recognition.src.evaluate import evaluate_model
from recognition.src import scorecard as sc
from recognition.src.config import MODELS_DIR
from pathlib import Path
import pandas as pd

# only the fingerspelling CNNs (skip any word_* sequence models)
rows = [evaluate_model(p.stem, p, test_ds, class_names)
        for p in sorted(Path(MODELS_DIR).glob('*.keras')) if not p.stem.startswith('word_')]
ranked = sc.score(rows)
pd.DataFrame(ranked)[['model', 'accuracy', 'macro_f1', 'latency', 'size', 'total']]

## 7 · Trust check + export the winner
**Grad-CAM** confirms each model looks at the *hand* (not the background) — update its `robustness` score from the heatmaps, add `stability` from the live desktop test, then re-run cell 6. Finally, export the winner to TFLite / TF.js / keras and download it from the **Files** pane into your local `recognition/models/`.

In [ ]:
winner = ranked[0]['model']
print('winner:', winner)
!python -m recognition.src.xai_gradcam --model recognition/models/{winner}.keras --n 8
!python -m recognition.src.export --model recognition/models/{winner}.keras --target all